In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "alessandro.macchi6@studenti.unimi.it"
os.environ['KAGGLE_KEY'] = "UniProjects26"
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows

HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/GetDatasetMetadata (Caused by NameResolutionError("HTTPSConnection(host='api.kaggle.com', port=443): Failed to resolve 'api.kaggle.com' ([Errno 11001] getaddrinfo failed)"))


In [2]:
import zipfile

zip_file = "imdb-dataset-of-top-1000-movies-and-tv-shows.zip"
extraction_path = "imdb_data"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Extracted files: {os.listdir(extraction_path)}")

Extracted files: ['imdb-dataset-of-top-1000-movies-and-tv-shows.zip', 'imdb_top_1000.csv']


In [46]:
import pandas as pd

df = pd.read_csv('imdb_data/imdb_top_1000.csv')

# Implement the Subsampling Toggle
USE_SUBSAMPLE = True
if USE_SUBSAMPLE:
    df = df.sample(frac=0.2, random_state=6)

# Select the star columns
stars_df = df[['Star1', 'Star2', 'Star3', 'Star4']]

# Convert to a list of lists (baskets), dropping NaN values
baskets = []
for index, row in stars_df.iterrows():
    # Drop missing values and convert to a list
    basket = [star for star in row if pd.notna(star)]
    if basket: # Only add if the basket isn't empty
        baskets.append(basket)

In [43]:
def apriori(baskets, support_threshold):
    
    baskets = list(baskets) # otherwise it gets exhausted after the first pass

    # ==========================================
    # FIRST PASS
    # ==========================================
    names2int = {}
    id_counter = 1
    item_counts = {}

    for basket in baskets:
        for star in basket:
            if star not in names2int:
                names2int[star] = id_counter
                id_counter += 1
                item_counts[names2int[star]] = 1
            else:
                item_counts[names2int[star]] += 1

    # ==========================================
    # BETWEEN TWO PASSES
    # ==========================================
    freq_items = {}
    new_numbering = 1

    for item in item_counts:
        if item_counts[item] >= support_threshold:
            freq_items[item] = new_numbering
            new_numbering += 1
        else:
            freq_items[item] = 0

    # ==========================================
    # SECOND PASS
    # ==========================================
    pairs_freq_items = []
    for basket in baskets:
        temp_freq = []
        for item in basket:
            if freq_items[names2int[item]] != 0:
                temp_freq.append(names2int[item])

        n = len(temp_freq)
        for i in range(n):
            for j in range(i + 1, n):
                # with min/max each pair is sorted in ascending order to avoid that (A, B) =/ (B, A)
                item1 = min(temp_freq[i], temp_freq[j])
                item2 = max(temp_freq[i], temp_freq[j])

                pairs_freq_items.append((item1, item2))

        temp_freq.clear()

    counts_of_pairs = {}

    for pair in pairs_freq_items:
        if pair not in counts_of_pairs:
            counts_of_pairs[pair] = 0
        counts_of_pairs[pair] += 1

    frequent_pairs = []

    for pair, count in counts_of_pairs.items():
        if count >= support_threshold:
            actor1 = next(name for name, integer in names2int.items() if integer == pair[0]) # to go back to names
            actor2 = next(name for name, integer in names2int.items() if integer == pair[1])

            frequent_pairs.append((actor1, actor2))

    return frequent_pairs

In [54]:
imdb_freq_stars_pairs = apriori(baskets, support_threshold=3)